# Piotroski F-Score Quality Value - Research Notebook

**Source**: QC Strategy Library #343 - High Book-to-Market High F-Score Quality Value

**Concept**: A fundamental value strategy using the Piotroski F-Score (9-point accounting-based scoring
system) to identify financially strong value stocks. Selects top 20% by book-to-market, filters for
F-Score >= 8, equal-weighted portfolio with monthly rebalance.

In [1]:
try:
    from AlgorithmImports import *
    qb = QuantBook()
    QC_ENV = True
except (ImportError, NameError):
    import pandas as pd
    import numpy as np
    from datetime import datetime, timedelta
    QC_ENV = False
    qb = None

if QC_ENV:
    qb.AddEquity("SPY", Resolution.Daily)
    print(f"Strategy: Value + Quality screen")
    print(f"Universe: Top 20% book-to-market, F-Score >= 8")
    print(f"Architecture: Alpha Framework (5 components)")
    print(f"Rebalance: Monthly (month start)")
    print(f"Warmup: 3 years (fundamental data accumulation)")
else:
    print("Local environment - using yfinance fallback for data")

Local environment - using yfinance fallback for data


## Strategy Logic

### Piotroski F-Score (9 components)

The F-Score is a 9-point accounting-based scoring system developed by Joseph Piotroski (2000).
Each component earns 0 or 1 point, for a total score of 0-9.

**Profitability (4 points)**:
1. **ROA > 0**: Positive return on assets
2. **Operating Cash Flow > 0**: Positive cash generation
3. **ROA improving**: Current year ROA > prior year ROA
4. **Accruals quality**: CFO/Assets > ROA (cash flow exceeds reported earnings)

**Leverage, Liquidity, Source of Funds (3 points)**:
5. **Leverage decreasing**: Lower long-term debt to assets ratio
6. **Liquidity improving**: Higher current ratio
7. **No equity issuance**: Shares outstanding not increasing

**Operating Efficiency (2 points)**:
8. **Gross margin improving**: Better pricing power
9. **Asset turnover improving**: More efficient asset utilization

### Selection Pipeline
1. Filter for liquid stocks (price > $1, volume > $100K)
2. Rank by book-to-market (1/P/B ratio), keep top 20% (value stocks)
3. Calculate F-Score using 3-year RollingWindow of fundamentals
4. Select stocks with F-Score >= threshold (default 8)
5. Cap at max_universe_size (default 100)
6. Equal-weight all selected stocks

In [2]:
# Fetch SPY benchmark data - handle QC Cloud MemoizingEnumerable and local yfinance fallback
if QC_ENV:
    spy_raw = qb.History("SPY", timedelta(252 * 12), Resolution.Daily)
    if isinstance(spy_raw, pd.DataFrame):
        spy_hist = spy_raw
    else:
        spy_hist = pd.DataFrame([{'close': float(b.Close), 'open': float(b.Open),
                                   'high': float(b.High), 'low': float(b.Low),
                                   'volume': float(b.Volume)} for b in spy_raw])
else:
    import yfinance as yf
    spy_hist = yf.Ticker("SPY").history(period="12y", auto_adjust=True).reset_index()
    spy_hist.columns = [str(c).lower() for c in spy_hist.columns]

print(f"SPY data: {len(spy_hist)} days")

spy_close = spy_hist["close"]
spy_returns = spy_close.pct_change().dropna()

print(f"\nSPY benchmark statistics (12-year):")
print(f"  Annualized return: {(1 + spy_returns.mean())**252 - 1:.2%}")
print(f"  Annualized vol: {spy_returns.std() * (252**0.5):.2%}")
print(f"  Sharpe (rf=0): {(spy_returns.mean() / spy_returns.std()) * (252**0.5):.2f}")

SPY data: 3018 days

SPY benchmark statistics (12-year):
  Annualized return: 15.76%
  Annualized vol: 17.39%
  Sharpe (rf=0): 0.84


In [3]:
# Simulate F-Score distribution analysis
import numpy as np

np.random.seed(42)
n_stocks = 500

# Simulate F-Scores (roughly normally distributed around 5)
f_scores = np.clip(np.random.normal(4.5, 2.0, n_stocks), 0, 9).astype(int)

print("Simulated F-Score distribution (500 stocks):")
for score in range(10):
    count = (f_scores == score).sum()
    pct = count / n_stocks * 100
    bar = "#" * int(pct)
    print(f"  Score {score}: {count:3d} ({pct:5.1f}%) {bar}")

high_score = (f_scores >= 8).sum()
print(f"\nStocks with F-Score >= 8: {high_score} ({high_score/n_stocks*100:.1f}%)")
print(f"After top 20% value filter: ~{int(high_score * 0.2)} stocks")

Simulated F-Score distribution (500 stocks):
  Score 0:  15 (  3.0%) ###
  Score 1:  31 (  6.2%) ######
  Score 2:  71 ( 14.2%) ##############
  Score 3:  79 ( 15.8%) ###############
  Score 4: 105 ( 21.0%) #####################
  Score 5:  90 ( 18.0%) ##################
  Score 6:  56 ( 11.2%) ###########
  Score 7:  31 (  6.2%) ######
  Score 8:  17 (  3.4%) ###
  Score 9:   5 (  1.0%) #

Stocks with F-Score >= 8: 22 (4.4%)
After top 20% value filter: ~4 stocks


## Key Observations

- **Piotroski F-Score** (2000) captures financial health across profitability, leverage, and efficiency
- **Value + Quality interaction**: combining cheap stocks (high B/M) with financial strength outperforms
  either screen alone, addressing the "value trap" problem
- **Alpha Framework** separation of concerns: universe, alpha, portfolio, execution are independent
- **3-year warmup** needed because F-Score requires 3 years of fundamental data (current, prior, 2-year lag)

### Strengths
- Academic grounding (Piotroski 2000, "Value Investing: The Use of Historical Financial Statement
  Information to Separate Winners from Losers")
- 9 independent signals reduce noise of any single metric
- Alpha Framework architecture is modular and extensible
- Equal weighting avoids concentration risk in large caps
- SpreadExecutionModel protects against illiquid entries

### Risks
- Fundamental data has reporting lag (10K filings may be months old)
- Value traps: high B/M may reflect genuine distress, not mispricing
- Monthly rebalance may be too slow for rapid regime changes
- No risk management (stop losses, position limits) beyond quality screening
- 3-year warmup period before any trading begins
- F-Score >= 8 threshold may be too restrictive in some market conditions

In [4]:
# Backtest results placeholder
print("=" * 50)
print("BACKTEST RESULTS (Author Reported)")
print("=" * 50)
print("  OOS 1Y Sharpe: 2.09")
print("  5Y CAGR: 18.44%")
print("  5Y Drawdown: 24.20%")
print("  Win Rate: 62%")
print()
print("Run backtest via QC MCP to verify:")
print("  create_compile -> create_backtest -> read_backtest")
print()
print("Architecture:")
print("  Alpha Framework: Universe + Alpha + Portfolio + Execution")
print("  5 Python files: main, universe, symbol_data, piotroski_score, piotroski_factors")
print("  Warmup: 3 years (fundamental data)")
print("  Universe: Hourly resolution, monthly rebalance")

BACKTEST RESULTS (Author Reported)
  OOS 1Y Sharpe: 2.09
  5Y CAGR: 18.44%
  5Y Drawdown: 24.20%
  Win Rate: 62%

Run backtest via QC MCP to verify:
  create_compile -> create_backtest -> read_backtest

Architecture:
  Alpha Framework: Universe + Alpha + Portfolio + Execution
  5 Python files: main, universe, symbol_data, piotroski_score, piotroski_factors
  Warmup: 3 years (fundamental data)
  Universe: Hourly resolution, monthly rebalance
